# Modern evaluation and comparison with Laura’s classical pipeline
Quantitative evaluation can run without Laura’s files. Joint topic matching and team review require validated classical exports. Use identical reference text preprocessing for both models; LDA mixture probabilities and HDBSCAN membership strengths have different meanings.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'projectproposal.tex').exists())
ROOT = REPO/'modern-methods/project'
sys.path.insert(0, str(ROOT))
from src import topic_helpers as h
OUT = ROOT/'Results/modern'


In [2]:
df, embeddings = h.load_inputs(ROOT)
selected = json.loads((OUT/'selected_run.json').read_text())
folder = OUT/'runs'/selected['run_id']
model = h.load_model(folder)
modern = pd.read_csv(folder/'assignments.csv',dtype={'article_id':str})
terms = pd.read_csv(folder/'terms.csv')
texts, dictionary = h.reference_texts(df)
metrics, per_topic = h.evaluate(model, embeddings, texts, dictionary)
h.write_json(OUT/'evaluation.json',metrics)
per_topic.to_csv(OUT/'topic_coherence.csv',index=False)
display(metrics)
counts, proportions = h.publication_exports(df, modern, OUT)

/Users/marykate/Desktop/F26/DataMiningProject/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'topic_count': 70,
 'outlier_fraction': 0.17118713370587432,
 'coherence': 0.7009812800660455,
 'vocabulary_coverage': 1.0,
 'keyword_diversity': 0.6328571428571429,
 'silhouette_cosine': 0.09502828866243362,
 'silhouette_status': 'ok',
 'silhouette_umap': 0.6134032011032104,
 'silhouette_umap_status': 'ok',
 'topic_sizes': {-1: 1256,
  0: 542,
  1: 441,
  2: 339,
  3: 315,
  4: 243,
  5: 224,
  6: 173,
  7: 161,
  8: 137,
  9: 126,
  10: 121,
  11: 119,
  12: 118,
  13: 108,
  14: 108,
  15: 108,
  16: 107,
  17: 95,
  18: 94,
  19: 92,
  20: 92,
  21: 85,
  22: 80,
  23: 76,
  24: 75,
  25: 73,
  26: 72,
  27: 70,
  28: 66,
  29: 64,
  30: 63,
  31: 60,
  32: 59,
  33: 54,
  34: 54,
  35: 51,
  36: 50,
  37: 50,
  38: 47,
  39: 47,
  40: 47,
  41: 45,
  42: 45,
  43: 43,
  44: 42,
  45: 42,
  46: 41,
  47: 40,
  48: 36,
  49: 35,
  50: 35,
  51: 34,
  52: 34,
  53: 34,
  54: 33,
  55: 33,
  56: 32,
  57: 32,
  58: 32,
  59: 32,
  60: 31,
  61: 30,
  62: 30,
  63: 27,
  64: 27,
  65:

## Classical exports
Place `assignments.csv`, `terms.csv`, `topics.csv`, and `manifest.json` in `Results/classical/`. See the contract in the notebook README. A missing handoff is reported without blocking modern metrics; malformed or misaligned files raise an error.

In [3]:
COMPARISON = ROOT/'Results/comparison'; COMPARISON.mkdir(parents=True,exist_ok=True)
CLASSICAL = ROOT/'Results/classical'
required = ['assignments.csv','terms.csv','topics.csv','manifest.json']
missing = [name for name in required if not (CLASSICAL/name).exists()]
classical = classical_terms = classical_topics = None
if missing:
    print('Joint comparison pending; missing classical files:', ', '.join(missing))
else:
    classical, classical_terms, classical_topics = h.validate_classical(CLASSICAL, df.article_id.tolist())
    manifest = json.loads((CLASSICAL/'manifest.json').read_text())
    assert manifest['corpus_hash'] == h.digest(df[['article_id','text']].values.tolist()), 'Classical corpus differs'
    classical_coherence = h.coherence(classical_terms,texts,dictionary)
    classical_coherence.to_csv(COMPARISON/'classical_coherence.csv',index=False)
    display(classical_coherence)
    matrix = h.jaccard_table(terms,classical_terms)
    matrix.to_csv(COMPARISON/'jaccard.csv')
    import matplotlib.pyplot as plt
    fig,ax=plt.subplots(figsize=(12,8))
    image=ax.imshow(matrix,aspect='auto',vmin=0,vmax=1)
    ax.set_yticks(range(len(matrix)),matrix.index); ax.set_xticks(range(len(matrix.columns)),matrix.columns,rotation=90)
    ax.set_xlabel('Classical topic'); ax.set_ylabel('BERTopic topic')
    fig.colorbar(image,ax=ax,label='Top-ten term Jaccard similarity'); fig.tight_layout()
    fig.savefig(COMPARISON/'jaccard.png',dpi=180)
    h.publication_exports(df,classical,COMPARISON/'classical')
    display(manifest)
    modern_row = dict(model='bertopic', **{k:v for k,v in metrics.items() if k!='topic_sizes'})
    top = classical_terms[(classical_terms.topic_id>=0)&(classical_terms['rank']<=10)]
    classical_silhouette, status = h.silhouette(embeddings,classical.topic_id.to_numpy(),'cosine')
    classical_row = dict(model=str(classical.model.iloc[0]),topic_count=classical.topic_id.nunique(),outlier_fraction=float(classical.topic_id.eq(-1).mean()),coherence=classical_coherence.coherence.mean(),vocabulary_coverage=classical_coherence.vocabulary_coverage.mean(),keyword_diversity=top.term.nunique()/len(top) if len(top) else np.nan,silhouette_cosine=classical_silhouette,silhouette_status=status)
    pd.DataFrame([modern_row,classical_row]).to_csv(COMPARISON/'model_metrics.csv',index=False)
    joined = modern[['article_id','topic_id']].merge(classical[['article_id','topic_id']],on='article_id',validate='one_to_one',suffixes=('_modern','_classical'))
    pd.crosstab(joined.topic_id_modern,joined.topic_id_classical).to_csv(COMPARISON/'assignment_contingency.csv')
    # Hardware and stage boundaries must be checked before interpreting these values.
    runtime = [{'model':'bertopic',**json.loads((folder/'manifest.json').read_text())['metrics']}, {'model':str(classical.model.iloc[0]),**manifest.get('timings',{})}]
    pd.DataFrame(runtime).to_csv(COMPARISON/'runtime_comparison.csv',index=False)


Joint comparison pending; missing classical files: assignments.csv, terms.csv, topics.csv, manifest.json


## Freeze labels and generate review packets
Two team members independently score article–topic fit 1–5 and flag irrelevant articles or ambiguous labels. The 100 random articles are separate from model-selection evidence. Joint review files are created only once both models are available. Twenty outliers are inspected separately.

In [4]:
labels = pd.read_csv(OUT/'topic_labels.csv')
non_outliers = labels.topic_id.ge(0)
assert labels.topic_id.is_unique and set(labels.topic_id) == set(model.topics_), 'Label IDs do not match selected model'
labels_ready = labels.loc[non_outliers,['manual_label','evidence_notes']].fillna('').apply(lambda col:col.str.strip().ne('')).all().all()
if not labels_ready:
    print('Human review pending: complete manual labels and evidence notes.')
else:
    final_labels={int(r.topic_id):r.manual_label for r in labels[non_outliers].itertuples()}
    final_labels[-1]='Outlier / unassigned'
    model.set_topic_labels(final_labels)
    topic_summary=pd.read_csv(folder/'topics.csv')
    topic_summary['label']=topic_summary.topic_id.map(final_labels)
    topic_summary.to_csv(OUT/'final_topics.csv',index=False)
    if classical is None:
        print('Joint human review pending: Laura’s exports are missing.')
if labels_ready and classical is not None:
    lookup = {('bertopic',int(r.topic_id)):r.manual_label for r in labels[non_outliers].itertuples()}
    lookup.update({(str(classical.model.iloc[0]),int(r.topic_id)):r.label for r in classical_topics.itertuples()})
    packet = h.review_packets(df,modern,classical,COMPARISON,lookup)
    display(packet.drop(columns='text').head())
    display(h.summarize_reviews(COMPARISON))

Human review pending: complete manual labels and evidence notes.


## Report interpretation
Discuss topic quality, outlier coverage, seed stability, runtime, and reviewer agreement together. Silhouette is conditional on assigned articles; UMAP distances can exaggerate separation. Jaccard measures lexical overlap, not semantic equivalence. Report publication proportions with their denominators and corpus-selection bias.

Use run manifests to compare stage times on the same hardware. Include first model-load/download costs and cached embedding timing separately; do not call cached runtime a cold end-to-end run. The final report must explicitly mark any missing human or classical evaluation.

Sources: project proposal; lecture outline and bibliography; Chapter 5 notebook; [BERTopic paper](https://arxiv.org/abs/2203.05794); [Sentence-BERT](https://aclanthology.org/D19-1410/); [GTE model card](https://huggingface.co/thenlper/gte-small); [Gensim coherence](https://radimrehurek.com/gensim/models/coherencemodel.html).